# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. Ba tín hiệu an toàn, mỗi tín hiệu một bài test nhỏ và
một phán quyết. Tất cả đều đo trong **khung dự báo**: đứng ở thời điểm ra quyết định, tín hiệu này có
tách được nhóm sẽ suy giảm trong 30 ngày kế tiếp hay không.

## 1. Distributions

Nhìn trước khi quyết định. Ba điều phải nhớ với dữ liệu tìm kiếm:

- **Đuôi cực nặng.** Impressions phân bố theo luật lũy thừa: trung vị vài trăm, tối đa hàng trăm nghìn.
  Đó là lý do mọi cột traffic đều vào mô hình dưới dạng `log1p`.
- **Rate là phần trăm ×100.** `ctr = 0.76` nghĩa là 0.76%, không phải 76%.
- **`avg_position = 0` nghĩa là "không có dữ liệu"**, không phải hạng 0 — 1,205 dòng như vậy.

In [1]:
# --- Bootstrap: chạy được cả ở local lẫn trên Colab ---
import os, sys, urllib.request

BRANCHES = [
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main",
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/claude/search-ranking-capstone-166z1j",
]

def _pipeline_dir() -> str:
    """Toàn bộ logic capstone nằm trong MỘT file: work/scripts/capstone_pipeline.py."""
    for p in ["../scripts", "work/scripts", "scripts", "../../work/scripts"]:
        if os.path.exists(os.path.join(p, "capstone_pipeline.py")):
            return p
    os.makedirs("work/scripts", exist_ok=True)
    for base in BRANCHES:
        try:
            urllib.request.urlretrieve(f"{base}/work/scripts/capstone_pipeline.py",
                                       "work/scripts/capstone_pipeline.py")
            return "work/scripts"
        except Exception:
            continue
    raise RuntimeError("Không tải được capstone_pipeline.py")

sys.path.insert(0, _pipeline_dir())
import numpy as np
import pandas as pd
import capstone_pipeline as cp

raw = cp.load_raw()
frame = cp.build_frame(raw)
d, population = cp.apply_population_filter(frame)
y = d["label_declined"].to_numpy()
groups = d["client_id"].to_numpy()
X = cp.design_matrix(d)

print(f"Population: {population['rows_modelled']:,} trang / {population['clients_modelled']} client "
      f"(lọc từ {population['rows_start']:,} dòng, ngưỡng impressions_prev_30d >= {cp.MIN_PREV_IMPRESSIONS})")
print(f"Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): {y.mean():.4f}")

cols = ["impressions_prev_30d", "impressions_first30", "clicks_prev_30d",
        "prior_ctr", "prior_impr_trend_pct", "content_age_days_at_decision", "word_count"]
desc = d[cols].describe(percentiles=[.1, .25, .5, .75, .9, .99]).T.round(2)
display(desc)

skew = (d["impressions_prev_30d"].max() / d["impressions_prev_30d"].median())
print(f"Đuôi nặng cỡ nào: impressions_prev_30d lớn nhất gấp {skew:,.0f}x trung vị "
      f"({d['impressions_prev_30d'].max():,.0f} so với {d['impressions_prev_30d'].median():,.0f}).")
print(f"=> log1p là bắt buộc, không phải trang trí.")
print(f"\nGotcha kiểm chứng: số dòng có avg_position = 0 (không có dữ liệu, không phải hạng 0) "
      f"trong dữ liệu thô = {(raw['avg_position'] == 0).sum():,}")

Population: 18,010 trang / 30 client (lọc từ 30,000 dòng, ngưỡng impressions_prev_30d >= 100)
Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): 0.6155


,count,mean,std,min,10%,25%,50%,75%,90%,99%,max
impressions_prev_30d,18010.0,2955.42,7718.36,100.00,158.00,293.00,797.00,2471.00,6782.20,35461.54,218786.00
impressions_first30,18010.0,3274.23,7604.12,0.00,185.00,415.00,1093.00,3046.75,7758.10,33447.47,258505.00
clicks_prev_30d,18010.0,9.01,36.16,0.00,0.00,0.00,1.00,5.00,19.00,132.00,1627.00
prior_ctr,18010.0,0.25,0.38,0.00,0.00,0.00,0.11,0.35,0.69,1.70,6.78
prior_impr_trend_pct,18010.0,-0.97,75.27,-98.83,-62.64,-45.68,-21.25,14.16,76.72,300.00,300.00
content_age_days_at_decision,18010.0,228.89,135.92,60.00,74.00,101.00,206.00,317.00,436.00,515.00,534.00
word_count,18010.0,3283.01,1289.11,692.00,2359.00,2760.00,2877.00,3321.00,5508.00,7377.00,9546.00


Đuôi nặng cỡ nào: impressions_prev_30d lớn nhất gấp 275x trung vị (218,786 so với 797).
=> log1p là bắt buộc, không phải trang trí.

Gotcha kiểm chứng: số dòng có avg_position = 0 (không có dữ liệu, không phải hạng 0) trong dữ liệu thô = 1,205


## 2. Signal test #1 / #2 / #3 (verdict each)

Ba giả thuyết, ba bài test, ba phán quyết — và hai trong ba lần dữ liệu **không** nói đúng điều tôi
chờ đợi. Phán quyết được viết *sau khi* nhìn bảng, không phải trước.

**Tín hiệu #1 — Động lượng trước đó.** Giả thuyết: trang đã mất nhu cầu giữa hai cửa sổ *nhìn thấy được*
sẽ tiếp tục mất trong cửa sổ kế tiếp. → **CONFIRMED, kèm một nếp gấp quan trọng.** Rủi ro giảm dần khi
động lượng cải thiện (70.7% → 52.4%), nhưng nhóm **bùng nổ > +50% lại quay đầu tăng lên 58.7%**. Quan hệ
không đơn điệu: trang vừa tăng vọt có xu hướng rơi trở lại. Đây chính là điều mô hình nhìn thấy khi nó
gán hệ số **dương** cho impressions cửa sổ trước.

**Tín hiệu #2 — Hiển thị cao mà CTR thấp.** Giả thuyết: trang có nhiều hiển thị nhưng CTR dưới trung vị
là nhóm rủi ro cao hơn. → **CONFIRMED, và mạnh hơn tôi tưởng.** Đơn điệu qua cả bốn tứ phân vị,
73.6% (CTR thấp nhất) so với 48.9% (CTR cao nhất) — chênh 24.7 điểm phần trăm, n ≈ 2,780 mỗi nhóm.

**Tín hiệu #3 — Bài dài thì bền hơn** (Finding #1 của paper FlyRank tháng 3/2026: trang tăng trưởng dài
hơn 37.6%). → **MIXED, và không sống sót khi vào mô hình.** Theo nhóm thì nhóm 1000–2000 từ suy giảm
nhiều hơn thật (79.4%, n=1,444, lift 1.29). Nhưng trung bình hai nhóm gần như bằng nhau (3,483 so với
3,432 từ, chênh 1.5%), nhóm `< 1000 từ` chỉ có **n = 15** nên không được đọc, và khi đứng cạnh các tín
hiệu nhu cầu/click thì `word_count` gần như không đóng góp gì (permutation importance ≈ 0.0009 PR-AUC).
Kết luận an toàn: độ dài bài **có liên hệ theo nhóm**, nhưng **không phải là thứ để ưu tiên theo**.

In [2]:
base = float(y.mean())
print(f"Base rate (mốc so sánh cho mọi bảng dưới đây): {base:.4f}\n")

# --- Tín hiệu #1: động lượng trước đó ---
d["_mom"] = pd.cut(d["prior_impr_trend_pct"], bins=[-101, -50, -20, 0, 20, 50, 301],
                   labels=["≤ -50%", "-50..-20%", "-20..0%", "0..+20%", "+20..+50%", "> +50%"])
t1 = d.groupby("_mom", observed=True)["label_declined"].agg(["mean", "count"]).round(4)
t1.columns = ["Tỷ lệ suy giảm", "n"]
t1["Lift vs base"] = (t1["Tỷ lệ suy giảm"] / base).round(2)
print("TÍN HIỆU #1 — động lượng impressions TRƯỚC thời điểm quyết định:")
display(t1)
falling, spiking = t1["Tỷ lệ suy giảm"].iloc[0], t1["Tỷ lệ suy giảm"].iloc[-1]
lowest = t1["Tỷ lệ suy giảm"].min()
print(f"VERDICT: CONFIRMED, kèm nếp gấp — nhóm tụt mạnh nhất suy giảm {falling:.1%}, "
      f"đáy ở {lowest:.1%}, nhưng nhóm bùng nổ > +50% quay lên {spiking:.1%}.")
print("   Quan hệ KHÔNG đơn điệu: trang vừa tăng vọt có xu hướng rơi trở lại (mean reversion).\n")

# --- Tín hiệu #2: hiển thị cao mà CTR thấp ---
hi = d[d["impressions_prev_30d"] >= 500].copy()
hi["_ctr_q"] = pd.qcut(hi["prior_ctr"].rank(method="first"), 4,
                       labels=["Q1 (CTR thấp nhất)", "Q2", "Q3", "Q4 (CTR cao nhất)"])
t2 = hi.groupby("_ctr_q", observed=True)["label_declined"].agg(["mean", "count"]).round(4)
t2.columns = ["Tỷ lệ suy giảm", "n"]
t2["Lift vs base"] = (t2["Tỷ lệ suy giảm"] / base).round(2)
print("TÍN HIỆU #2 — CTR cửa sổ trước, chỉ xét trang có >= 500 impressions:")
display(t2)
spread_pp = (t2["Tỷ lệ suy giảm"].max() - t2["Tỷ lệ suy giảm"].min()) * 100
print(f"VERDICT: CONFIRMED — đơn điệu qua cả 4 tứ phân vị, chênh {spread_pp:.1f} điểm phần trăm "
      f"({t2['Tỷ lệ suy giảm'].max():.1%} so với {t2['Tỷ lệ suy giảm'].min():.1%}), "
      f"n ≈ {int(t2['n'].min()):,} mỗi nhóm.")
print("   Đây là nền tảng của reason code low_ctr_high_exposure.\n")

# --- Tín hiệu #3: độ dài bài ---
wc = d[d["has_word_count"] == 1].copy()
wc["_wc"] = pd.cut(wc["word_count"], bins=[0, 1000, 2000, 3500, 100000],
                   labels=["< 1000", "1000-2000", "2000-3500", "3500+"])
t3 = wc.groupby("_wc", observed=True)["label_declined"].agg(["mean", "count"]).round(4)
t3.columns = ["Tỷ lệ suy giảm", "n"]
t3["Lift vs base"] = (t3["Tỷ lệ suy giảm"] / base).round(2)
print("TÍN HIỆU #3 — độ dài bài (Finding #1 của paper, đo lại trong khung dự báo):")
display(t3)
small = t3[t3["n"] < 50]
means = wc.groupby("label_declined")["word_count"].mean().round(0)
print(f"CẢNH BÁO cỡ mẫu: nhóm {list(small.index)} chỉ có n = {list(small['n'])} — không được đọc.")
print(f"Trung bình số từ: không suy giảm = {means.loc[0]:,.0f}, suy giảm = {means.loc[1]:,.0f} "
      f"(chênh {abs(means.loc[0]-means.loc[1])/means.loc[0]:.1%}) — paper báo cáo chênh 37.6% "
      "trên so sánh cắt ngang giữa hai đầu phân phối.")
print("VERDICT: MIXED — nhóm 1000-2000 từ suy giảm nhiều hơn thật (lift 1.29), nhưng trung bình hai "
      "nhóm gần như bằng nhau và word_count gần như không đóng góp gì khi đứng cạnh tín hiệu nhu cầu "
      "và click (permutation importance ≈ 0.0009 PR-AUC). Có liên hệ, nhưng không phải thứ để ưu tiên theo.")

Base rate (mốc so sánh cho mọi bảng dưới đây): 0.6155

TÍN HIỆU #1 — động lượng impressions TRƯỚC thời điểm quyết định:


,Tỷ lệ suy giảm,n,Lift vs base
_mom,,,
≤ -50%,0.7066,3735,1.15
-50..-20%,0.6462,5481,1.05
-20..0%,0.5625,2896,0.91
0..+20%,0.5386,1866,0.87
+20..+50%,0.5241,1538,0.85
> +50%,0.5874,2494,0.95


VERDICT: CONFIRMED, kèm nếp gấp — nhóm tụt mạnh nhất suy giảm 70.7%, đáy ở 52.4%, nhưng nhóm bùng nổ > +50% quay lên 58.7%.
   Quan hệ KHÔNG đơn điệu: trang vừa tăng vọt có xu hướng rơi trở lại (mean reversion).

TÍN HIỆU #2 — CTR cửa sổ trước, chỉ xét trang có >= 500 impressions:


,Tỷ lệ suy giảm,n,Lift vs base
_ctr_q,,,
Q1 (CTR thấp nhất),0.7360,2780,1.20
Q2,0.6385,2780,1.04
Q3,0.5678,2779,0.92
Q4 (CTR cao nhất),0.4888,2780,0.79


VERDICT: CONFIRMED — đơn điệu qua cả 4 tứ phân vị, chênh 24.7 điểm phần trăm (73.6% so với 48.9%), n ≈ 2,779 mỗi nhóm.
   Đây là nền tảng của reason code low_ctr_high_exposure.

TÍN HIỆU #3 — độ dài bài (Finding #1 của paper, đo lại trong khung dự báo):


,Tỷ lệ suy giảm,n,Lift vs base
_wc,,,
< 1000,0.4000,15,0.65
1000-2000,0.7936,1444,1.29
2000-3500,0.6279,7274,1.02
3500+,0.6851,4052,1.11


CẢNH BÁO cỡ mẫu: nhóm ['< 1000'] chỉ có n = [15] — không được đọc.
Trung bình số từ: không suy giảm = 3,483, suy giảm = 3,432 (chênh 1.5%) — paper báo cáo chênh 37.6% trên so sánh cắt ngang giữa hai đầu phân phối.
VERDICT: MIXED — nhóm 1000-2000 từ suy giảm nhiều hơn thật (lift 1.29), nhưng trung bình hai nhóm gần như bằng nhau và word_count gần như không đóng góp gì khi đứng cạnh tín hiệu nhu cầu và click (permutation importance ≈ 0.0009 PR-AUC). Có liên hệ, nhưng không phải thứ để ưu tiên theo.


## 3. The flag-linked test

FlyRank có nhóm cờ vận hành dựa trên **độ tươi nội dung** (`freshness_tier` từ
`days_since_last_update`), với giả định: *nội dung càng lâu không cập nhật thì càng dễ suy giảm*.

Bài test này cho ra hai kết luận chồng lên nhau, và cả hai đều quan trọng:

1. **Về mặt tương quan quan sát được:** dữ liệu cho kết quả **NGƯỢC** với giả định — nhóm được cập nhật
   gần đây nhất (0–30 ngày) lại có tỷ lệ suy giảm *cao hơn* nhóm 91–180 ngày.
2. **Về mặt phương pháp:** đừng vội đọc mục 1 như một phát hiện về nội dung. `days_since_last_update`
   được đo lúc export, và 68.3% số trang được cập nhật *bên trong* chính cửa sổ kết quả. Nhiều khả năng
   ta đang nhìn thấy **phản ứng của đội biên tập**: họ cập nhật đúng những trang đang tụt.
   Đó là quan hệ nhân quả ngược (reverse causation), không phải bằng chứng rằng refresh gây hại.

Kết luận dùng được: cột này **không hợp lệ làm feature dự báo**, và một cờ vận hành dựa trên nó cần được
tính lại với cửa sổ thời gian căn đúng trước khi tin.

In [3]:
t4 = raw.copy()
t4["label_declined"] = (t4["trend_pct"] < cp.DECLINE_THRESHOLD_PCT).astype(int)
tab = t4.groupby("freshness_tier")["label_declined"].agg(["mean", "count"]).round(4)
tab.columns = ["Tỷ lệ suy giảm", "n"]
print("Giả định của cờ: 'lâu không cập nhật => dễ suy giảm hơn'. Dữ liệu nói gì:")
display(tab[tab["n"] >= 50])

inside = int((raw["days_since_last_update"] < 30).sum())
print(f"VERDICT: OPPOSITE — nhưng KHÔNG dùng được làm phát hiện về nội dung.")
print(f"  {inside:,}/{len(raw):,} trang ({inside/len(raw):.1%}) được cập nhật BÊN TRONG cửa sổ kết quả.")
print("  Nhiều khả năng đây là phản ứng của biên tập viên (họ sửa đúng trang đang tụt), "
      "không phải tác động của việc refresh.")
print("  => Cột này bị loại khỏi feature set của capstone.")

Giả định của cờ: 'lâu không cập nhật => dễ suy giảm hơn'. Dữ liệu nói gì:


,Tỷ lệ suy giảm,n
freshness_tier,,
0-30,0.5113,20480
181+,0.4713,174
31-90,0.5886,175
91-180,0.6107,9171


VERDICT: OPPOSITE — nhưng KHÔNG dùng được làm phát hiện về nội dung.
  20,480/30,000 trang (68.3%) được cập nhật BÊN TRONG cửa sổ kết quả.
  Nhiều khả năng đây là phản ứng của biên tập viên (họ sửa đúng trang đang tụt), không phải tác động của việc refresh.
  => Cột này bị loại khỏi feature set của capstone.


## 4. What this means in practice

Ba câu cho một đội nội dung:

1. **Ưu tiên theo động lượng đã nhìn thấy được, đừng ưu tiên theo tuổi hay độ tươi.** Trang đã mất trên
   50% nhu cầu giữa hai cửa sổ trước đó suy giảm tiếp ở tỷ lệ 70.7% so với base rate 61.6%. Và đừng bỏ
   qua đầu bên kia: trang vừa tăng vọt trên +50% cũng quay đầu (58.7%) — một cú nhảy hiển thị không tự
   nó là tin tốt.
2. **CTR thấp trên trang hiển thị lớn là tín hiệu đáng tin nhất mà một biên tập viên có thể tự kiểm.**
   73.6% so với 48.9% giữa tứ phân vị CTR thấp nhất và cao nhất. Hành động ở đây là title và mô tả,
   không phải viết lại bài.
3. **Đừng viết dài thêm chỉ vì một biểu đồ nói bài dài thì tăng trưởng.** Liên hệ theo nhóm có tồn tại,
   nhưng độ dài bài gần như không còn giá trị dự báo khi đã biết nhu cầu và click. Nếu muốn mở rộng nội
   dung thì hãy làm vì lý do nội dung.

Ngôn ngữ: tất cả những gì trên đây là **quan sát được** trên một danh mục, trong một cửa sổ.
Không có thử nghiệm nào ở đây, nên không có câu nào nói rằng làm X sẽ tạo ra Y.

In [4]:
summary = pd.DataFrame([
    {"Tín hiệu": "Động lượng impressions trước đó", "Verdict": "CONFIRMED (không đơn điệu)",
     "Dùng thế nào": "Đặc trưng chính + reason code visibility_slipping; nhóm bùng nổ > +50% cũng là rủi ro"},
    {"Tín hiệu": "CTR thấp trên trang hiển thị cao", "Verdict": "CONFIRMED (24.7 điểm %)",
     "Dùng thế nào": "Reason code low_ctr_high_exposure — hành động là title/meta, không phải viết lại"},
    {"Tín hiệu": "Bài dài thì bền hơn", "Verdict": "MIXED — không sống sót trong mô hình",
     "Dùng thế nào": "Giữ trong feature set, không đưa vào khuyến nghị"},
    {"Tín hiệu": "Độ tươi nội dung (cờ vận hành)", "Verdict": "OPPOSITE + nhiễm bẩn cửa sổ",
     "Dùng thế nào": "Loại khỏi feature set; cần căn lại cửa sổ trước khi tin"},
])
display(summary)

,Tín hiệu,Verdict,Dùng thế nào
0,Động lượng impressions trước đó,CONFIRMED (không đơn điệu),Đặc trưng chính + reason code visibility_slipp...
1,CTR thấp trên trang hiển thị cao,CONFIRMED (24.7 điểm %),Reason code low_ctr_high_exposure — hành động ...
2,Bài dài thì bền hơn,MIXED — không sống sót trong mô hình,"Giữ trong feature set, không đưa vào khuyến nghị"
3,Độ tươi nội dung (cờ vận hành),OPPOSITE + nhiễm bẩn cửa sổ,Loại khỏi feature set; cần căn lại cửa sổ trướ...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.